# Notebook 03: Large-Scale PySpark Processing & Spark SQL Aggregations
## BDA Mini-Project: Air Pollution Analysis Across India

### Objectives:
- Demonstrate Apache Spark distributed architecture on 47M records.
- Join monitoring station geographical metadata (lat/lon) with measurement observations.
- Execute Spark SQL temporary view queries for city, seasonal, and diurnal metrics.
- Explain Hive partitioning, columnar Parquet compression, and performance metrics.

In [1]:
import sys, os
from pathlib import Path
PROJECT_ROOT = Path(os.getcwd()).resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

import json
import pandas as pd
import src.config as config
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, avg, count, round

spark = (
    SparkSession.builder
    .appName('BDA_03_PySpark_Processing')
    .master('local[*]')
    .config('spark.sql.shuffle.partitions', '4')
    .config('spark.ui.enabled', 'false')
    .getOrCreate()
)
print(f'Spark Version: {spark.version} | Master: {spark.sparkContext.master}')

Spark Version: 3.5.3 | Master: local[*]


### 1. Ingesting Clean Partitioned Parquet Storage

In [2]:
parquet_dir = PROJECT_ROOT / 'data' / 'processed' / 'clean_measurements_2024.parquet'
clean_files = [str(f.resolve()) for f in sorted(parquet_dir.rglob('*.parquet')) if f.name != '_SUCCESS']
df = spark.read.parquet(*clean_files)
print(f'Total Processed Observations in Hive Parquet: {df.count():,}')
df.printSchema()

Total Processed Observations in Hive Parquet: 46,937,146
root
 |-- station_id: string (nullable = true)
 |-- station_name: string (nullable = true)
 |-- state_name: string (nullable = true)
 |-- city_name: string (nullable = true)
 |-- parameter_name: string (nullable = true)
 |-- unit: string (nullable = true)
 |-- collected_at: timestamp (nullable = true)
 |-- value: double (nullable = true)
 |-- source: string (nullable = true)
 |-- obs_year: integer (nullable = true)
 |-- obs_day: integer (nullable = true)
 |-- obs_hour: integer (nullable = true)
 |-- day_of_week: string (nullable = true)
 |-- season: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)



### 2. Spark SQL Analytics Temporary View

In [3]:
df.createOrReplaceTempView('air_quality')

# Spark SQL Query 1: Top 10 States with highest PM2.5 averages across 2024
query_states = '''
SELECT 
    state_name,
    COUNT(*) as total_readings,
    ROUND(AVG(CASE WHEN parameter_name = 'PM2.5' THEN value END), 2) as annual_avg_pm25,
    ROUND(AVG(CASE WHEN parameter_name = 'PM10' THEN value END), 2) as annual_avg_pm10,
    ROUND(AVG(CASE WHEN parameter_name = 'NO2' THEN value END), 2) as annual_avg_no2
FROM air_quality
WHERE state_name IS NOT NULL AND state_name != ''
GROUP BY state_name
HAVING COUNT(CASE WHEN parameter_name = 'PM2.5' THEN 1 END) >= 5000
ORDER BY annual_avg_pm25 DESC
LIMIT 10
'''
df_state_top = spark.sql(query_states)
df_state_top.show(10, truncate=False)

+----------------+--------------+---------------+---------------+--------------+
|state_name      |total_readings|annual_avg_pm25|annual_avg_pm10|annual_avg_no2|
+----------------+--------------+---------------+---------------+--------------+
|Delhi           |3413013       |105.0          |212.61         |42.8          |
|Chandigarh      |351479        |70.5           |136.44         |28.34         |
|Haryana         |2755380       |68.39          |132.84         |19.14         |
|Himachal Pradesh|100322        |66.82          |147.27         |22.09         |
|Tripura         |89180         |64.42          |130.97         |8.79          |
|Bihar           |3347361       |63.44          |134.76         |16.63         |
|Assam           |800277        |57.2           |106.29         |13.33         |
|Punjab          |704362        |54.19          |116.93         |21.53         |
|West Bengal     |1819821       |54.04          |103.1          |25.12         |
|Uttar Pradesh   |4978242   

### 3. Pipeline Performance Metrics

In [4]:
metrics_path = PROJECT_ROOT / 'outputs' / 'metrics' / 'spark_pipeline_metrics.json'
with open(metrics_path) as f:
    p_metrics = json.load(f)
for k, v in p_metrics.items():
    print(f'{k:<25}: {v}')

dataset_year             : 2024
input_records            : 46937170
valid_records            : 46937146
filtered_records         : 24
retention_percentage     : 100.0
total_elapsed_seconds    : 236.29
spark_master             : local[*]
spark_version            : 3.5.3


In [5]:
spark.stop()
print('SparkSession closed.')

SparkSession closed.
